In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold, train_test_split

# All user settings are here. Use the folder containing these four files.
DIRECTORY = Path.cwd()  # Or Path(r"/full/path/to/Submission Files")
INPUT_FILE = DIRECTORY / "Input.csv"
OUTPUT_FILE = DIRECTORY / "Predictions.csv"

QUANTUM_NUMBERS = ["J", "v1", "v2", "v3", "r", "p"]
BAND_QUANTUM_NUMBERS = ["v1", "v2", "v3"]
STATE_KEY = ["ISO"] + QUANTUM_NUMBERS
MASS_FEATURES = ["m_C", "m_O_sum"]
FEATURES = QUANTUM_NUMBERS + MASS_FEATURES
PARITY_LABELS = {"e": 0, "f": 1}
C_MASS = {2: 12.0000000, 3: 13.00335484, 4: 14.0032420}
O_MASS = {6: 15.99491462, 7: 16.99913170, 8: 17.99915961}
DISTANCE_COLUMN = "MARVEL_Distance"

SEED = 2023
RANDOM_FOLDS = 5
VALIDATION_FRACTION = 0.20
EARLY_STOPPING_ROUNDS = 200
EXPORT_CHUNK_ROWS = 100_000
N_JOBS = min(8, len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1))
MODEL_PARAMS = {
    "objective": "huber", "alpha": 0.02,
    "n_estimators": 50_000, "learning_rate": 0.05,
    "max_depth": 4, "num_leaves": 127, "min_child_samples": 10,
    "subsample": 0.7, "subsample_freq": 0, "colsample_bytree": 1.0,
    "reg_alpha": 0.0, "reg_lambda": 0.0,
    "n_jobs": N_JOBS, "verbosity": -1,
}

In [ ]:
if INPUT_FILE.resolve() == OUTPUT_FILE.resolve():
    raise ValueError("Input and output must be different files.")
input_columns = pd.read_csv(INPUT_FILE, nrows=0).columns.tolist()
added_columns = ["WIDE_Predictions", "Ran_Predictions", DISTANCE_COLUMN]
if set(input_columns) & set(added_columns):
    raise ValueError("Use the input CSV, without prediction or distance columns.")

f = pd.read_csv(INPUT_FILE, usecols=STATE_KEY + ["EAmes", "EMARVEL"],
                low_memory=False, float_precision="round_trip")
for column in [c for c in f.columns if c != "p"]:
    f[column] = pd.to_numeric(f[column], errors="coerce")
parity = f.p.astype("string").str.strip().str.lower().map(PARITY_LABELS)
parity = parity.fillna(pd.to_numeric(f.p, errors="coerce"))
if not parity.isin([0, 1]).all():
    raise ValueError("Parity must be e/f or 0/1.")
f["p"] = parity.astype("int8")
keys = f[STATE_KEY].to_numpy(dtype=float)
if not np.isfinite(keys).all() or not np.equal(keys, np.floor(keys)).all():
    raise ValueError("Physical quantum numbers must be finite integers.")
f["ISO"] = f.ISO.astype(int)
if f.empty or f.duplicated(STATE_KEY).any():
    raise ValueError("Input must contain one row per physical level.")
if not np.isfinite(f.EAmes).all() or np.isinf(f.EMARVEL).any():
    raise ValueError("Ames energies must be finite; missing MARVEL energies must be blank.")

isotopes = sorted(f.ISO.unique())
# AFGL ISO = abc represents O(a)-C(b)-O(c). Masses are in u.
# m_C = C_MASS[b]; m_O_sum = O_MASS[a] + O_MASS[c].
# These features are calculated here, rather than read from Input.csv.
masses = {iso: (C_MASS[(iso // 10) % 10], O_MASS[iso // 100] + O_MASS[iso % 10])
          for iso in isotopes}
for position, column in enumerate(MASS_FEATURES):
    f[column] = f.ISO.map(lambda iso: masses[iso][position])
f["DeltaE"] = f.EMARVEL - f.EAmes
known = np.isfinite(f.EMARVEL.to_numpy(float))
measured_index = np.flatnonzero(known)
inferred_index = np.flatnonzero(~known)
if len(isotopes) < 2 or f.loc[known, "ISO"].nunique() < 2:
    raise ValueError("WIDE needs MARVEL levels in at least two isotopologues.")
if len(measured_index) < RANDOM_FOLDS or not np.isfinite(f[FEATURES]).all().all():
    raise ValueError("Insufficient MARVEL levels or invalid model features.")

In [ ]:
def fit_model(outer_train, seed):
    fit_index, stop_index = train_test_split(
        outer_train, test_size=VALIDATION_FRACTION, random_state=int(seed))
    model = lgb.LGBMRegressor(random_state=int(seed), **MODEL_PARAMS)
    model.fit(
        f.iloc[fit_index][FEATURES], f.iloc[fit_index].DeltaE,
        eval_X=f.iloc[stop_index][FEATURES],
        eval_y=f.iloc[stop_index].DeltaE,
        eval_metric="l1",
        callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)])
    return model


def predict_seed(mode, seed):
    prediction = np.full(len(f), np.nan)
    if mode == "WIDE":
        # The target isotopologue is excluded from fitting and early stopping.
        for iso in isotopes:
            outer_train = np.flatnonzero(known & f.ISO.ne(iso).to_numpy())
            outer_test = np.flatnonzero(f.ISO.eq(iso).to_numpy())
            model = fit_model(outer_train, seed)
            prediction[outer_test] = (f.iloc[outer_test].EAmes.to_numpy()
                                      + model.predict(f.iloc[outer_test][FEATURES]))
    elif mode == "Random":
        inferred_sum = np.zeros(len(inferred_index))
        splitter = KFold(n_splits=RANDOM_FOLDS, shuffle=True, random_state=int(seed))
        for train_positions, test_positions in splitter.split(measured_index):
            outer_train = measured_index[train_positions]
            outer_test = measured_index[test_positions]
            model = fit_model(outer_train, seed)
            prediction[outer_test] = (f.iloc[outer_test].EAmes.to_numpy()
                                      + model.predict(f.iloc[outer_test][FEATURES]))
            if len(inferred_index):
                inferred_sum += (f.iloc[inferred_index].EAmes.to_numpy()
                                 + model.predict(f.iloc[inferred_index][FEATURES]))
        prediction[inferred_index] = inferred_sum / RANDOM_FOLDS
    else:
        raise ValueError(mode)
    if not np.isfinite(prediction).all():
        raise ValueError("Every input level must receive a finite prediction.")
    return prediction


def marvel_band_distance():
    distance = np.full(len(f), "N/A", dtype=object)
    for iso in isotopes:
        mask = f.ISO.eq(iso).to_numpy()
        reference = f.loc[mask & known, BAND_QUANTUM_NUMBERS].drop_duplicates().to_numpy(dtype=np.int64)
        if not len(reference):
            continue  # Keep N/A when this isotopologue has no MARVEL reference.
        bands, inverse = np.unique(f.loc[mask, BAND_QUANTUM_NUMBERS].to_numpy(dtype=np.int64),
                                   axis=0, return_inverse=True)
        nearest = np.abs(bands[:, None, :] - reference[None, :, :]).sum(axis=2).min(axis=1)
        distance[mask] = nearest[inverse]
    return distance

In [ ]:
# Distance uses the supplied MARVEL coverage; it is not a model feature.
distance = marvel_band_distance()
wide_prediction = predict_seed("WIDE", SEED)
random_prediction = predict_seed("Random", SEED)

# Preserve every original column, cell value and row order.
temporary = OUTPUT_FILE.with_suffix(".csv.tmp")
rows_written = 0
for chunk in pd.read_csv(INPUT_FILE, dtype=str, keep_default_na=False,
                         chunksize=EXPORT_CHUNK_ROWS):
    stop = rows_written + len(chunk)
    chunk["WIDE_Predictions"] = wide_prediction[rows_written:stop]
    chunk["Ran_Predictions"] = random_prediction[rows_written:stop]
    chunk[DISTANCE_COLUMN] = distance[rows_written:stop]
    chunk.to_csv(temporary, index=False, mode="w" if rows_written == 0 else "a",
                 header=rows_written == 0, float_format="%.17g")
    rows_written = stop
if rows_written != len(f):
    raise ValueError("The input row count changed during the run.")
temporary.replace(OUTPUT_FILE)

In [ ]:
# Read the saved predictions; this cell does not retrain models.
results = pd.read_csv(OUTPUT_FILE,
    usecols=["ISO", "EMARVEL", "WIDE_Predictions", "Ran_Predictions"],
    float_precision="round_trip")
records = []
for iso in [*sorted(results.ISO.unique()), "Total"]:
    part = results if iso == "Total" else results.loc[results.ISO.eq(iso)]
    measured = part.loc[np.isfinite(part.EMARVEL)]
    records.append({
        "ISO": iso,
        "Total_levels": len(part),
        "Known_levels": len(measured),
        "Unmeasured_levels": len(part) - len(measured),
        "Total_predictions": int(np.isfinite(part[["WIDE_Predictions", "Ran_Predictions"]]).to_numpy().sum()),
        "MARVEL_available_%": 100 * len(measured) / len(part),
        "WIDE_MAE_cm-1": (measured.WIDE_Predictions - measured.EMARVEL).abs().mean(),
        "Random_MAE_cm-1": (measured.Ran_Predictions - measured.EMARVEL).abs().mean(),
    })
summary = pd.DataFrame(records)
print(f"MARVEL reference; seed {SEED}. MAEs use only MARVEL-known levels.")
print("Total MAEs pool absolute errors over all known levels.")
print("Total_predictions counts both methods: two predictions per input level.")
print(summary.to_string(index=False, na_rep="N/A", float_format=lambda value: f"{value:.6f}"))